# Prompt regression suite

Prompt là code: mỗi lần sửa một câu trong system prompt, đổi model hay đổi `temperature`, chất lượng có thể tụt mà
không ai biết. Bạn dựng một **bộ kiểm hồi quy cho prompt**: prompt có phiên bản, tập câu ghim, cache theo khoá xác
định, thước đo kèm khoảng tin cậy, và một **cổng** quyết định "được merge hay không" so với baseline. Rồi bạn dùng nó
cho bốn thay đổi thật và viết `REPORT.md`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp, hằng. Test gọi `macro_f1`, `paired_bootstrap`, `cache_key`,
  `regression_gate` trên dữ liệu nhỏ; giữ nguyên tên, tham số, `LABELS`, `GateResult`.
- Phần còn lại (định nghĩa phiên bản prompt, chạy, cache, in báo cáo) là thiết kế của bạn, trong cell `explore`.
- Cần Ollama với `qwen2.5:3b`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/11-llm-apis-prompting/exercises/advanced/01-prompt-regression-suite -q`

In [ ]:
# explore: tìm gốc repo, tải UIT-VSFC nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

VSFC = ensure("uit-vsfc")
CACHE = VSFC / "cache"
CACHE.mkdir(exist_ok=True)

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import hashlib
import json
from dataclasses import dataclass, field

import numpy as np

## 1. Thước đo (dán từ bài Intermediate 02)

`macro_f1` và `paired_bootstrap` đúng như đặc tả của bài Intermediate 02 (notebook tự chứa, không import chéo bài).

In [ ]:
LABELS = ["negative", "neutral", "positive"]


def macro_f1(y_true: list[str], y_pred: list[str | None]) -> float:
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError


def paired_bootstrap(y_true, pred_a, pred_b, n_boot: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    # TODO: dán lời giải bài Intermediate 02
    raise NotImplementedError

## 2. Khoá cache và cổng hồi quy

`cache_key(prompt, model, params, item_id) -> str`: khoá xác định cho một lời gọi, để chạy lại suite không gọi lại
model khi không có gì đổi, và **tự** gọi lại khi bất kỳ thứ gì ảnh hưởng tới output đổi. Đúng định nghĩa này (test so
từng ký tự): SHA-256 (chữ hex thường, 64 ký tự) của chuỗi UTF-8
`json.dumps({"prompt": prompt, "model": model, "params": params, "item": item_id}, sort_keys=True,
ensure_ascii=False, separators=(",", ":"))`. `prompt` có thể là chuỗi hoặc dict/list (ví dụ cả danh sách messages
mẫu).

`regression_gate(y_true, base_pred, cand_pred, *, max_drop=0.02, class_max_drop=0.10, max_unparsed_increase=0,
n_boot=2000, seed=0) -> GateResult`: so ứng viên với baseline trên cùng câu. `diff, lo, hi` là kết quả
`paired_bootstrap(y_true, base_pred, cand_pred, n_boot, seed)` (ứng viên trừ baseline). `reasons` theo đúng thứ tự:

1. `"significant_drop"` nếu `hi < 0` (khoảng tin cậy nằm hẳn dưới 0);
2. `"point_drop"` nếu `diff < -max_drop`;
3. `"class_drop:<nhãn>"` cho từng nhãn theo thứ tự `LABELS` mà F1 của nhãn đó tụt **hơn** `class_max_drop`;
4. `"more_unparsed"` nếu số `None` của ứng viên trừ của baseline lớn hơn `max_unparsed_increase`.

`passed` là `True` khi không có lý do nào. Cổng này cố ý **không** đòi ứng viên phải tốt hơn có ý nghĩa thống kê: trên
150 câu, gần như không thay đổi nào qua được tiêu chuẩn đó, và cổng sẽ chặn mọi thứ. Nó chặn tụt rõ ràng, tụt quá
ngưỡng, tụt ở một lớp (lớp hiếm hay bị hy sinh mà macro không lộ rõ), và output hỏng nhiều hơn.

In [ ]:
@dataclass
class GateResult:
    passed: bool
    diff: float
    lo: float
    hi: float
    reasons: list[str] = field(default_factory=list)


def cache_key(prompt, model: str, params: dict, item_id: str) -> str:
    """SHA-256 hex của JSON chuẩn hoá (định nghĩa ở trên)."""
    # TODO
    raise NotImplementedError


def regression_gate(y_true, base_pred, cand_pred, *, max_drop: float = 0.02, class_max_drop: float = 0.10,
                    max_unparsed_increase: int = 0, n_boot: int = 2000, seed: int = 0) -> GateResult:
    """Cổng hồi quy của ứng viên so với baseline (định nghĩa ở trên)."""
    # TODO
    raise NotImplementedError

## 3. Suite của bạn

Thiết kế trong các cell `explore` dưới đây (khung gợi ý, sửa tuỳ ý):

- **Phiên bản prompt** là dữ liệu, không phải chuỗi rải rác: một dict `{tên phiên bản: {"messages": ..., "format":
  ..., "parse": ...}}` hoặc tệp YAML; baseline là `zero_shot` của bài Intermediate 02.
- **Tập ghim** 150 câu như bài Intermediate 02 (hoặc lớn hơn, ghi rõ).
- **Chạy có cache** theo `cache_key`; lưu `reply`, giây, token vào, token ra cho từng câu.
- **Bốn ứng viên** so với baseline: (1) few-shot, (2) JSON schema, (3) CoT-lite, (4) một "sửa vội" mà bạn nghĩ vô hại,
  ví dụ bỏ dòng định nghĩa nhãn neutral khỏi system prompt hoặc đổi thứ tự nhãn. Thêm (5) tuỳ chọn: cùng prompt
  baseline, model khác hoặc chỉ CPU, để đo **sàn nhiễu**.
- **Báo cáo** bảng markdown: phiên bản, macro-F1 [CI], F1 từng lớp, `None`, p50 giây, token vào/ra, chi phí ước lượng
  cho 1 000 câu theo một bảng giá bạn chọn (ghi ngày kiểm hoặc ghi "giả định"), quyết định của cổng và lý do.

In [ ]:
# explore: khung suite (viết tiếp)
PROMPTS = {}           # tên phiên bản -> định nghĩa
BASELINE = "zero_shot"
# TODO (explore): dữ liệu ghim, chạy có cache theo cache_key, bảng kết quả, regression_gate cho từng ứng viên